# Notebook 01: Data Ingestion
## Load 947 CSVs → Unified Parquet Dataset
Source: MIT UV-Vis dataset (Rajeev J. Ram, Sci Rep 2025, DOI: 10.1038/s41598-024-83114-y)

In [ ]:
import os, re, struct
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')

BASE = Path('/run/media/sham/AI_/ai-stack/projects/biopharma-contamination-detection')
DATA_DIR = BASE / 'Bacteria Contamination Work'
OUT = BASE / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)

## Discover all 947 CSV files

In [ ]:
categories = {
    'sterile': list((DATA_DIR / 'Sterile samples').rglob('*.csv')),
    'contaminated': list((DATA_DIR / 'Contaminated samples').rglob('*.csv')),
    'timepoint': list((DATA_DIR / 'Timepoint Experiment').rglob('*.csv')),
}
all_csvs = sum(categories.values(), [])
for cat, files in categories.items():
    print(f'  {cat}: {len(files)} files')
print(f'  Total: {len(all_csvs)} files')

## Parse metadata from filenames

In [ ]:
def parse_meta(fp):
    name = fp.stem
    m = {'filepath': str(fp), 'filename': name}
    rel = str(fp.relative_to(DATA_DIR))
    if rel.startswith('Sterile'): m['category'], m['label'] = 'sterile', 0
    elif rel.startswith('Contaminated'): m['category'], m['label'] = 'contaminated', 1
    elif rel.startswith('Timepoint'): m['category'] = 'timepoint'
    else: m['category'] = 'unknown'
    
    # Organism mapping
    orgs = {'EColi':'Escherichia coli','SAureus':'Staphylococcus aureus','BSubtilis':'Bacillus subtilis',
            'CAlbicans':'Candida albicans','CSporogenes':'Clostridium sporogenes',
            'PAeruginosa':'Pseudomonas aeruginosa','PA_':'Pseudomonas aeruginosa',
            'SA_':'Staphylococcus aureus','BS_':'Bacillus subtilis','CA_':'Candida albicans',
            'CS_':'Clostridium sporogenes','CAcnes':'Cutibacterium acnes','CAc_':'Cutibacterium acnes',
            'PositiveControl':'Escherichia coli','NegativeControl':'None (control)',
            'PBSspiked':'PBS control','MSC_LZ':'MSC media control'}
    m['organism'] = 'Unknown'
    for pat, org in orgs.items():
        if pat in name: m['organism'] = org; break
    
    cfu = re.search(r'(\d+)CFU', name)
    m['cfu'] = int(cfu.group(1)) if cfu else 0
    donor = re.search(r'LZMSC(\d+)P(\d+)', name)
    m['donor_id'] = f"MSC{donor.group(1)}P{donor.group(2)}" if donor else ('D5' if 'D5' in name else 'Unknown')
    dt = re.search(r'(\d{8})', name)
    m['experiment_date'] = dt.group(1) if dt else ''
    tp = re.search(r'_(\d+)h_', name)
    m['timepoint_hours'] = int(tp.group(1)) if tp else None
    return m

for f in all_csvs[:5]:
    m = parse_meta(f)
    print(f"{f.name[:55]:55s} → {m['organism']:25s} CFU={m['cfu']:5d} {m['donor_id']}")

## Load all CSVs into unified DataFrame

In [ ]:
records, failed = [], []
for fp in tqdm(all_csvs, desc='Loading'):
    try:
        m = parse_meta(fp)
        df = pd.read_csv(fp, skiprows=1)
        if 'Wavelength' in df.columns and 'CorrectedAbs' in df.columns:
            wl, ab = df['Wavelength'].values, df['CorrectedAbs'].values
        elif 'Wavelength' in df.columns and 'Absorbance' in df.columns:
            wl, ab = df['Wavelength'].values, df['Absorbance'].values
        else:
            wl, ab = df.iloc[:,0].values, df.iloc[:,1].values
        m.update({'n_wl': len(wl), 'wl_min': wl.min(), 'wl_max': wl.max(),
                  'abs_min': ab.min(), 'abs_max': ab.max(), 'abs_mean': ab.mean(),
                  'wavelengths': wl.tolist(), 'absorbance': ab.tolist()})
        records.append(m)
    except Exception as e:
        failed.append({'file': str(fp), 'error': str(e)})

print(f'Loaded: {len(records)}, Failed: {len(failed)}')

## Build and inspect unified DataFrame

In [ ]:
df = pd.DataFrame(records)
print(f'Shape: {df.shape}')
print(f'\nBy category:\n{df["category"].value_counts().to_string()}')
print(f'\nBy organism:\n{df["organism"].value_counts().to_string()}')
print(f'\nBy CFU:\n{df["cfu"].value_counts().sort_index().to_string()}')
print(f'\nBy donor:\n{df["donor_id"].value_counts().head(15).to_string()}')

## Save to Parquet

In [ ]:
# Pack spectra as bytes for efficient storage
def pack(wl, ab):
    pairs = list(zip(wl, ab))
    return struct.pack(f'{len(pairs)}dd', *[v for p in pairs for v in p])

df['spectrum_bytes'] = df.apply(lambda r: pack(r['wavelengths'], r['absorbance']), axis=1)
save_cols = [c for c in df.columns if c not in ['wavelengths', 'absorbance']]
df_save = df[save_cols].copy()

out_path = OUT / 'real_dataset.parquet'
df_save.to_parquet(out_path, engine='pyarrow')
print(f'Saved: {out_path}')
print(f'Size: {os.path.getsize(out_path)/1024/1024:.1f} MB')

# Save metadata CSV (no spectra, quick reference)
meta_cols = ['filename','category','organism','cfu','label','donor_id','experiment_date',
             'timepoint_hours','n_wl','wl_min','wl_max','abs_min','abs_max','abs_mean']
df[[c for c in meta_cols if c in df.columns]].to_csv(OUT / 'dataset_metadata.csv', index=False)
print(f'Saved metadata CSV')

# Verify reload
chk = pd.read_parquet(out_path)
print(f'\nReloaded: {chk.shape}')
print(f'Label distribution: {chk["label"].value_counts().to_dict()}')
print('✅ Data ingestion complete!')